# 03_ml
Captured 2026-11-21 21:38 UTC

**Late-PO model beats the baseline on the time-split holdout:** PR-AUC 0.562 vs 0.455 (lift 0.107, bar 0.05, result: model adds value); recall in top 20% 0.347 vs 0.302. Scores feed gold.days_of_cover and disruption impact.


In [1]:
-- training run (run_log/train_late_po.json)
-- src/ai/train_late_po.py

{
 "task": "train_late_po",
 "mlflow_run_id": "6eed5d2c35af445dbe691ba6a7698bd8",
 "model_version": "6",
 "endpoint": "late-po-model",
 "endpoint_status": "updating",
 "holdout_start": "2026-06-12",
 "train_rows": 36988,
 "test_rows": 3793,
 "test_late_rate": 0.342,
 "pr_auc_model": 0.562,
 "pr_auc_baseline": 0.455,
 "roc_auc_model": 0.713,
 "roc_auc_baseline": 0.651,
 "recall_top20_model": 0.347,
 "recall_top20_baseline": 0.302,
 "pr_auc_lift": 0.107,
 "claim": "model adds value"
}

In [2]:
-- scoring run (run_log/score_po_risk.json)
-- src/ai/score_po_risk.py

{
  "task": "score_po_risk",
  "model_version": "6",
  "open_po_lines_scored": 4594,
  "mean_p_late": 0.355,
  "lines_p_late_over_0_5": 958,
  "mean_baseline_p_late": 0.336
}

In [3]:
-- serving endpoint state
databricks serving-endpoints get late-po-model

{"state": {"config_update": "NOT_UPDATING", "ready": "READY"}, "served": [["supply_chain.ml.late_po_model", "6"]]}

In [4]:
-- endpoint response: risky vs reliable vendor
databricks serving-endpoints query late-po-model

{"error": "Error: Post \"https://adb-7405619902341965.5.azuredatabricks.net/serving-endpoints/late-po-model/invocations\": request timed out after 1m0s of inactivity"}

In [5]:
-- model output in the score table
SELECT ... FROM ml.po_risk_score

mean_model_p_late | mean_baseline_p_late | high_risk_lines | open_lines
--------------------------------------------------------------------------
0.355             | 0.336                | 958             | 4594      
(1 rows)

In [6]:
-- GenAI: ai_query explanations + groundedness (run_log/explain_impact.json)
-- src/ai/explain_impact.py

{
  "task": "explain_impact",
  "llm": "databricks-claude-haiku-5-5",
  "explanations": 25,
  "grounded": 25,
  "ungrounded_skus": [],
  "sample": [
    {
      "sku_id": "SKU-0118",
      "explanation": "SKU-0118 (morphine, EU) depends on a single API source, M-API-015, and has 73.2 days of cover now, which drops to 52.3 days without that vendor, with a projected stockout on 2027-01-11 and a supply gap of 127.7 days. Qualifying an alternate source takes 180 days, so the gap cannot close unless qualification starts now."
    },
    {
      "sku_id": "SKU-0119",
      "explanation": "SKU-0119 (morphine, US) is single-sourced from API M-API-015, with 75.5 days of cover today that would drop to 62.1 days without the vendor, and it projects a stockout on 2027-01-21 with a 117.9-day supply gap. Because qualifying an alternate source takes 180 days, the next step is to start that qualification now."
    },
    {
      "sku_id": "SKU-0083",
      "explanation": "SKU-0083 (midazolam, APAC) rel